# Wan 2.2 fence video (Kaggle free GPU) -- A14B high-quality edition

**Before running:** In the right sidebar, set `Accelerator` to `GPU T4 x2` (or `GPU P100`) and turn `Internet` **On**. Then `Run All`.

This does NOT need the browser tab to stay open once you click **Save Version > Save & Run All (Commit)** -- it runs on Kaggle's servers in the background. The finished video appears in the notebook's **Output** tab when the commit finishes.

Uses the full Wan2.2 I2V-A14B model (14B params, high-noise + low-noise experts) for noticeably better quality than the earlier 5B version -- takes longer, but no image upload needed and no time constraint.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import torch
print('torch', torch.__version__, 'cuda available:', torch.cuda.is_available())
major, minor = (int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
if (major, minor) < (2, 7):
    print('Upgrading torch...')
    !pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
    print('If this ran, restart the kernel and re-run from the top.')

In [ ]:
# /kaggle/working is capped at 20GB (it's what gets persisted as Output).
# ~25GB of A14B model weights won't fit there -- use /kaggle/tmp instead,
# which gives ~60GB of scratch space that just doesn't persist after the
# run (fine, since only the final mp4 needs to survive).
import os
os.makedirs('/kaggle/tmp', exist_ok=True)
%cd /kaggle/tmp
!git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
%cd /kaggle/tmp/ComfyUI
!pip install -q -r requirements.txt

In [ ]:
%cd /kaggle/tmp/ComfyUI/custom_nodes
!git clone --depth 1 https://github.com/city96/ComfyUI-GGUF.git
!pip install -q -r ComfyUI-GGUF/requirements.txt
%cd /kaggle/tmp/ComfyUI

In [ ]:
import os
import urllib.request

os.makedirs('models/unet', exist_ok=True)
os.makedirs('models/text_encoders', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)
os.makedirs('input', exist_ok=True)

def expected_size(url):
    req = urllib.request.Request(url, method='HEAD')
    with urllib.request.urlopen(req) as r:
        return int(r.headers.get('Content-Length', -1))

# Wan2.2 I2V-A14B: the full 14B-parameter MoE model (high-noise + low-noise
# experts), a real step up from the 5B TI2V hybrid we used before. Q4_K_M is
# a good quality/size balance -- ~9.6GB per expert, ~19GB+6GB text encoder.
files_to_check = [
    ('models/unet/Wan2.2-I2V-A14B-HighNoise-Q4_K_M.gguf', 'https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-I2V-A14B-HighNoise-Q4_K_M.gguf'),
    ('models/unet/Wan2.2-I2V-A14B-LowNoise-Q4_K_M.gguf', 'https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q4_K_M.gguf'),
    ('models/text_encoders/umt5-xxl-encoder-Q8_0.gguf', 'https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q8_0.gguf'),
    ('models/vae/Wan2.1_VAE.safetensors', 'https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/VAE/Wan2.1_VAE.safetensors'),
]

for path, url in files_to_check:
    exp = expected_size(url)
    if os.path.exists(path) and os.path.getsize(path) == exp:
        print(f'{path}: already present ({exp} bytes) -- skipping download')
        continue
    print(f'Downloading {path} ...')
    os.system(f'wget -q --show-progress -O "{path}" "{url}"')
    actual = os.path.getsize(path)
    status = 'OK' if actual == exp else 'MISMATCH -- re-download this file!'
    print(f'{path}: {actual} / {exp} expected -- {status}')

In [ ]:
# Fetch the 2 source images from GitHub -- no manual upload needed.
# (Only first and last frame are used now -- one continuous generation.)
!wget -q -O input/start_no_fence.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/start%20frame.png"
!wget -q -O input/end_full_fence.png "https://raw.githubusercontent.com/bhogaljasdeep/wan22-fence-inputs/master/end%20frame%20with%20fence.png"
!ls -la input/
print('Images ready in input/')

In [ ]:
import subprocess, time, urllib.request

try:
    urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
    print('ComfyUI already running')
except Exception:
    proc = subprocess.Popen(
        ['python', 'main.py'],
        stdout=open('/kaggle/tmp/comfyui.log', 'w'), stderr=subprocess.STDOUT,
    )
    for _ in range(60):
        try:
            urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
            print('ComfyUI is up')
            break
        except Exception:
            time.sleep(2)
    else:
        print('ComfyUI did not start -- check /kaggle/tmp/comfyui.log')
        !tail -n 60 /kaggle/tmp/comfyui.log

In [ ]:
import json, time, urllib.request

SERVER = "http://127.0.0.1:8188"
UNET_HIGH = "Wan2.2-I2V-A14B-HighNoise-Q4_K_M.gguf"
UNET_LOW = "Wan2.2-I2V-A14B-LowNoise-Q4_K_M.gguf"
CLIP = "umt5-xxl-encoder-Q8_0.gguf"
VAE = "Wan2.1_VAE.safetensors"

# SMOKE TEST: run this first with tiny settings to prove the whole graph
# (node names, output indices, tiled decode, ffmpeg conversion) actually
# works end-to-end -- takes a few minutes instead of several hours. Once
# a smoke-test commit succeeds and produces a real (if tiny/ugly) mp4 in
# the Output tab, flip this to False and re-commit for the real run.
SMOKE_TEST = False

if SMOKE_TEST:
    WIDTH, HEIGHT, LENGTH = 512, 288, 17   # ~0.7s, tiny
    STEPS_TOTAL, STEPS_SPLIT = 8, 4
else:
    WIDTH, HEIGHT, LENGTH = 832, 480, 25  # ~4s at 6fps (testing, v2 prompt, 832x480)
    STEPS_TOTAL, STEPS_SPLIT = 40, 20

CFG = 5.0
SHIFT = 8.0
SEED = 42

# Single prompt covering the whole 5s transformation -- pillars descend first,
# then wire hooks on -- since we're going straight from the no-fence image to
# the complete-fence image in one generation (the mid/pillars image was only
# useful for stitching two segments, which we no longer need).
POSITIVE_PROMPT = (
    "aerial drone shot flying forward over farmland village at sunset, camera "
    "continuously moving forward the entire time. Tall wooden fence pillars "
    "are clearly visible falling and descending from high in the sky, dropping "
    "straight down in perfect symmetrical rows, a slow controlled descent that "
    "is clearly visible over several seconds, not appearing suddenly. As each "
    "pillar reaches the ground it plants into the earth and rights itself, "
    "rotating and straightening upright into a standing vertical fence post, "
    "one after another, forming a straight line of upright posts stretching "
    "into the distance. Then, barbed wire unspools and stretches itself "
    "between the pillars, hooking onto each upright post one by one in "
    "sequence from near to far, wire strands pulling taut across the line of "
    "posts. magical stop-motion construction, cinematic lighting, photorealistic"
)
NEGATIVE_PROMPT = "blurry, low quality, distorted, flickering, artifacts, watermark, text, static camera, jerky motion, falling, dropping, crashing, impact, bouncing"

def build_graph(start_image, end_image, filename_prefix, positive_prompt):
    return {
        "unet_loader_high": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": UNET_HIGH}},
        "unet_loader_low": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": UNET_LOW}},
        "clip_loader": {"class_type": "CLIPLoaderGGUF", "inputs": {"clip_name": CLIP, "type": "wan"}},
        "vae_loader": {"class_type": "VAELoader", "inputs": {"vae_name": VAE}},
        "model_sampling_high": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["unet_loader_high", 0], "shift": SHIFT}},
        "model_sampling_low": {"class_type": "ModelSamplingSD3", "inputs": {"model": ["unet_loader_low", 0], "shift": SHIFT}},
        "positive": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["clip_loader", 0], "text": positive_prompt}},
        "negative": {"class_type": "CLIPTextEncode", "inputs": {"clip": ["clip_loader", 0], "text": NEGATIVE_PROMPT}},
        "load_start": {"class_type": "LoadImage", "inputs": {"image": start_image}},
        "load_end": {"class_type": "LoadImage", "inputs": {"image": end_image}},
        # Core ComfyUI node (works with any Wan I2V-style checkpoint, unlike
        # the Wan2.2-5B-specific FLF2V node we needed before).
        "flf": {
            "class_type": "WanFirstLastFrameToVideo",
            "inputs": {
                "positive": ["positive", 0], "negative": ["negative", 0], "vae": ["vae_loader", 0],
                "width": WIDTH, "height": HEIGHT, "length": LENGTH, "batch_size": 1,
                "start_image": ["load_start", 0], "end_image": ["load_end", 0],
            },
        },
        # Stage 1: high-noise expert, steps 0 -> STEPS_SPLIT, leaves noise for stage 2.
        "ksampler_high": {
            "class_type": "KSamplerAdvanced",
            "inputs": {
                "model": ["model_sampling_high", 0], "positive": ["flf", 0], "negative": ["flf", 1],
                "latent_image": ["flf", 2], "add_noise": "enable", "noise_seed": SEED,
                "steps": STEPS_TOTAL, "cfg": CFG, "sampler_name": "uni_pc", "scheduler": "simple",
                "start_at_step": 0, "end_at_step": STEPS_SPLIT, "return_with_leftover_noise": "enable",
            },
        },
        # Stage 2: low-noise expert, continues from where stage 1 left off.
        "ksampler_low": {
            "class_type": "KSamplerAdvanced",
            "inputs": {
                "model": ["model_sampling_low", 0], "positive": ["flf", 0], "negative": ["flf", 1],
                "latent_image": ["ksampler_high", 0], "add_noise": "disable", "noise_seed": SEED,
                "steps": STEPS_TOTAL, "cfg": CFG, "sampler_name": "uni_pc", "scheduler": "simple",
                "start_at_step": STEPS_SPLIT, "end_at_step": 10000, "return_with_leftover_noise": "disable",
            },
        },
        # Tiled decode -- avoids the VRAM spike that killed the process on
        # Colab's T4 right after sampling finished with the 5B model.
        "vae_decode": {
            "class_type": "VAEDecodeTiled",
            "inputs": {
                "samples": ["ksampler_low", 0], "vae": ["vae_loader", 0],
                "tile_size": 256, "overlap": 64, "temporal_size": 32, "temporal_overlap": 8,
            },
        },
        "save_video": {
            "class_type": "SaveWEBM",
            "inputs": {"images": ["vae_decode", 0], "filename_prefix": filename_prefix, "codec": "vp9", "fps": 6.0, "crf": 20.0},
        },
    }

def queue_prompt(graph):
    data = json.dumps({"prompt": graph}).encode("utf-8")
    req = urllib.request.Request(f"{SERVER}/prompt", data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        return json.loads(resp.read())

# BUG FIX: this was capped at 14400s (4h) -- the A14B full-quality run
# legitimately took longer than that and our own polling loop would have
# killed it with an uncaught TimeoutError (which is likely why earlier runs
# produced "no output" even when ComfyUI itself hadn't crashed). Kaggle
# sessions cap out around 9h, so give ourselves margin under that.
def wait_for_completion(prompt_id, poll_interval=10, timeout=32000):
    start = time.time()
    while time.time() - start < timeout:
        with urllib.request.urlopen(f"{SERVER}/history/{prompt_id}") as resp:
            hist = json.loads(resp.read())
        if prompt_id in hist:
            entry = hist[prompt_id]
            status = entry.get("status", {})
            if status.get("completed"):
                return entry
            if status.get("status_str") == "error":
                raise RuntimeError(f"Prompt {prompt_id} failed: {json.dumps(status, indent=2)}")
        elapsed = int(time.time() - start)
        if elapsed % 60 < poll_interval:
            print(f"  ...still running ({elapsed}s elapsed)")
        time.sleep(poll_interval)
    raise TimeoutError(f"Prompt {prompt_id} did not complete within {timeout}s")

def run_segment(name, start_image, end_image, filename_prefix, positive_prompt):
    print(f"=== Queuing segment: {name} ===")
    graph = build_graph(start_image, end_image, filename_prefix, positive_prompt)
    result = queue_prompt(graph)
    prompt_id = result["prompt_id"]
    print(f"  prompt_id={prompt_id}")
    entry = wait_for_completion(prompt_id)
    outputs = entry.get("outputs", {})
    video_info = outputs.get("save_video", {})
    print(f"  DONE: {json.dumps(video_info)}")
    return video_info

print(f'Ready ({"SMOKE TEST" if SMOKE_TEST else "FULL QUALITY"} mode). Run the next cell to generate.')

In [ ]:
result = run_segment("no-fence -> complete fence (one continuous 5s clip)", "start_no_fence.png", "end_full_fence.png", "fence_full", POSITIVE_PROMPT)
print("DONE")

In [ ]:
# Convert to mp4 and copy ONLY the small final file into /kaggle/working
# (the 20GB-capped, persisted directory) -- everything else stays in
# /kaggle/tmp and is discarded automatically when the session ends.
%cd /kaggle/tmp/ComfyUI/output
!ls -la *.webm

webm_file = [f for f in __import__('os').listdir('.') if f.startswith('fence_full')][0]
!ffmpeg -y -i "{webm_file}" -c:v libx264 -pix_fmt yuv420p -crf 18 /kaggle/working/fence_full_5s.mp4
print('Final video at /kaggle/working/fence_full_5s.mp4 -- check the Output tab after commit.')